# 第2章 — フィッシングサイトと迷惑メールの検出

『セキュリティエンジニアのための機械学習』§2.2–2.3の独立した再実装。
**ランタイム → すべてのセルを実行**。CPUで数分が目安（環境により変動）。
第1章の事前実行、GitHub認証、手動ファイル配置は不要。

| 対象 | 入力 | モデル | 陽性（1） |
|---|---|---|---|
| フィッシング | UCIの30個の既存特徴量 | ロジスティック回帰・決定木 | 元ラベル -1 |
| 迷惑メール | Enron1の英語メール | TF-IDF + LightGBM | spamフォルダ |

URLを入力して特徴量を収集する実装ではない。実サイトへの接続やメール送信は行わない。
データの配布元・固定コミット・SHA-256をコードに記載する。

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *[
            "numpy==2.2.6",
            "pandas==2.2.3",
            "scipy==1.15.3",
            "scikit-learn==1.7.2",
            "matplotlib==3.10.3",
            "optuna==4.5.0",
            "lightgbm==4.6.0",
        ],
    ],
    check=True,
)

In [ ]:
import platform
from importlib.metadata import version

import matplotlib.pyplot as plt

print(f"Python {platform.python_version()} | CPU runtime")
for package in ("numpy", "pandas", "scipy", "scikit-learn", "optuna", "lightgbm"):
    print(f"{package}: {version(package)}")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 1. データ取得と監査

出版社の公開リポジトリからデータだけを取得する。SHA-256不一致は停止し、ZIPは展開せず読む。
空データを除き、同じ入力でラベルが矛盾する行は全件除外、残りの重複は1件にまとめる。
メールの重複判定は大文字小文字と空白の違いを無視する。本文自体は改変しない。

**UCIの -1 はフィッシング、+1 は正常**。陽性が危険側を示すよう0/1に変換する。
同じ特徴量の行を分割間にまたがせないため、書籍の数値とは直接比較できない。

In [ ]:
from dataclasses import dataclass
from hashlib import sha256
from pathlib import Path, PurePosixPath
from tempfile import NamedTemporaryFile
from urllib.request import urlopen
from zipfile import ZipFile

import numpy as np
import pandas as pd

UPSTREAM_REVISION = "565def0834d063281ff0776918ad829020919752"
DATA_URL = (
    f"https://raw.githubusercontent.com/oreilly-japan/ml-security-jp/{UPSTREAM_REVISION}/ch02"
)
DATA_HASHES = {
    "dataset.csv": "98395653cdd0e5a79c73bc59907e63ce82681d40947858b73464c08981a3629c",
    "enron1.zip": "97d1e3942057e7a9192af73adc70dad1f4279a05dadde2c1606a364c245519ed",
}
FEATURE_NAMES = (
    "having_IP_Address",
    "URL_Length",
    "Shortining_Service",
    "having_At_Symbol",
    "double_slash_redirecting",
    "Prefix_Suffix",
    "having_Sub_Domain",
    "SSLfinal_State",
    "Domain_registeration_length",
    "Favicon",
    "port",
    "HTTPS_token",
    "Request_URL",
    "URL_of_Anchor",
    "Links_in_tags",
    "SFH",
    "Submitting_to_email",
    "Abnormal_URL",
    "Redirect",
    "on_mouseover",
    "RightClick",
    "popUpWidnow",
    "Iframe",
    "age_of_domain",
    "DNSRecord",
    "web_traffic",
    "Page_Rank",
    "Google_Index",
    "Links_pointing_to_page",
    "Statistical_report",
)

In [ ]:
def fetch_dataset(name: str, cache_dir: Path) -> Path:
    expected = DATA_HASHES[name]
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    target = cache_dir / name
    if target.exists():
        if sha256(target.read_bytes()).hexdigest() != expected:
            raise ValueError(f"Checksum mismatch: {target}. Remove the cached file and retry.")
        return target

    temporary = None
    try:
        with urlopen(f"{DATA_URL}/{name}", timeout=60) as response:
            with NamedTemporaryFile(dir=cache_dir, delete=False) as stream:
                temporary = Path(stream.name)
                digest = sha256()
                size = 0
                while chunk := response.read(1024 * 1024):
                    size += len(chunk)
                    if size > 32 * 1024 * 1024:
                        raise ValueError("Dataset exceeds the download limit")
                    digest.update(chunk)
                    stream.write(chunk)
        if digest.hexdigest() != expected:
            raise ValueError(f"Checksum mismatch: {name}")
        temporary.replace(target)
        return target
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)

In [ ]:
@dataclass(frozen=True)
class DataAudit:
    raw_rows: int
    empty_rows: int
    conflicting_rows: int
    duplicate_rows: int
    retained_rows: int

In [ ]:
def deduplicate(frame: pd.DataFrame, keys: list[str]) -> tuple[pd.DataFrame, DataAudit]:
    frame = frame.copy()
    raw_rows = len(frame)
    empty = frame[keys].isna().any(axis=1)
    if keys == ["text"]:
        frame["_identity"] = frame["text"].fillna("").str.lower().str.split().str.join(" ")
        empty |= frame["_identity"].eq("")
        identity = ["_identity"]
    else:
        identity = keys
    frame = frame.loc[~empty].copy()
    conflicting = frame.groupby(identity, dropna=False)["label"].transform("nunique").gt(1)
    clean = frame.loc[~conflicting].drop_duplicates(identity)
    audit = DataAudit(
        raw_rows,
        int(empty.sum()),
        int(conflicting.sum()),
        len(frame) - int(conflicting.sum()) - len(clean),
        len(clean),
    )
    return clean.drop(columns="_identity", errors="ignore").reset_index(drop=True), audit

In [ ]:
def load_phishing(path: Path) -> tuple[pd.DataFrame, DataAudit]:
    values = pd.read_csv(path, header=None)
    if values.shape[1] != 31 or values.empty:
        raise ValueError("Expected 30 phishing features and one label")
    if values.isna().any().any() or not values.isin([-1, 0, 1]).all().all():
        raise ValueError("Invalid phishing feature values")
    if set(values.iloc[:, -1].unique()) != {-1, 1}:
        raise ValueError("Expected UCI labels -1 (phishing) and +1 (legitimate)")
    values.columns = [*FEATURE_NAMES, "label"]
    values = values.astype(np.int8)
    values["label"] = values["label"].eq(-1).astype(np.int8)
    return deduplicate(values, list(FEATURE_NAMES))

In [ ]:
def load_enron(path: Path) -> tuple[pd.DataFrame, DataAudit]:
    rows = []
    with ZipFile(path) as archive:
        members = archive.infolist()
        if len(members) > 10_000 or sum(m.file_size for m in members) > 64 * 1024 * 1024:
            raise ValueError("Enron archive exceeds the size limit")
        for member in sorted(members, key=lambda item: item.filename):
            if member.is_dir():
                continue
            if member.filename == "enron1/Summary.txt":
                continue
            parts = PurePosixPath(member.filename).parts
            if len(parts) != 3 or parts[0] != "enron1" or parts[1] not in {"ham", "spam"}:
                raise ValueError(f"Unexpected archive member: {member.filename}")
            if member.file_size > 1024 * 1024:
                raise ValueError(f"Oversized email: {member.filename}")
            text = archive.read(member).decode("utf-8", errors="replace")
            rows.append({"text": text, "label": int(parts[1] == "spam")})
    if not rows or {row["label"] for row in rows} != {0, 1}:
        raise ValueError("Enron archive must contain both ham and spam")
    return deduplicate(pd.DataFrame(rows), ["text"])

In [ ]:
cache_dir = Path("data")
phishing, phishing_audit = load_phishing(fetch_dataset("dataset.csv", cache_dir))
emails, email_audit = load_enron(fetch_dataset("enron1.zip", cache_dir))
display(pd.DataFrame({"phishing": vars(phishing_audit), "enron1": vars(email_audit)}).T)
display(
    pd.DataFrame(
        {
            "phishing": phishing.label.value_counts(),
            "enron1": emails.label.value_counts(),
        }
    ).rename(index={0: "legitimate / ham", 1: "phishing / spam"})
)

## 2. 訓練・検証・テストの分離

60%を訓練、20%を閾値選択、20%を最終テストに使い、クラス比率を維持する。
Optunaは訓練部分の層化3分割CVでF1を最大化する。標準化とTF-IDFの語彙・IDFも各fold内で学習する。
探索の並列化は行わず、乱数と試行回数を固定する。TF-IDFは疎行列のままLightGBMへ渡す。

In [ ]:
from typing import Any

import optuna
from lightgbm import LGBMClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from threadpoolctl import threadpool_limits

In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    seed: int = 101
    trials: int = 8
    folds: int = 3
    max_false_positive_rate: float = 0.01

    def __post_init__(self):
        if self.trials < 1 or self.folds < 2:
            raise ValueError("trials >= 1 and folds >= 2 are required")
        if not 0 <= self.max_false_positive_rate <= 1:
            raise ValueError("max_false_positive_rate must lie in [0, 1]")

In [ ]:
@dataclass(frozen=True)
class Split:
    train: np.ndarray
    validation: np.ndarray
    test: np.ndarray

In [ ]:
def split_indices(y, seed: int) -> Split:
    indices = np.arange(len(y))
    train_validation, test = train_test_split(
        indices,
        test_size=0.2,
        random_state=seed,
        stratify=y,
    )
    train, validation = train_test_split(
        train_validation,
        test_size=0.25,
        random_state=seed,
        stratify=np.asarray(y)[train_validation],
    )
    return Split(train, validation, test)

In [ ]:
def make_model(kind: str, params: dict[str, Any], seed: int) -> Pipeline:
    if kind == "logistic":
        return Pipeline(
            [
                ("scale", StandardScaler()),
                (
                    "classifier",
                    LogisticRegression(
                        C=params["C"],
                        solver="lbfgs",
                        max_iter=2000,
                        random_state=seed,
                    ),
                ),
            ]
        )
    if kind == "tree":
        return Pipeline([("classifier", DecisionTreeClassifier(**params, random_state=seed))])
    if kind == "spam":
        return Pipeline(
            [
                (
                    "tfidf",
                    TfidfVectorizer(
                        stop_words="english",
                        min_df=2,
                        max_features=20_000,
                        sublinear_tf=True,
                        dtype=np.float32,
                    ),
                ),
                (
                    "classifier",
                    LGBMClassifier(
                        **params,
                        objective="binary",
                        verbosity=-1,
                        n_jobs=2,
                        random_state=seed,
                        deterministic=True,
                        force_col_wise=True,
                        importance_type="gain",
                    ),
                ),
            ]
        )
    raise ValueError(f"Unknown model: {kind}")

In [ ]:
def suggest_params(trial: optuna.Trial, kind: str) -> dict[str, Any]:
    if kind == "logistic":
        return {"C": trial.suggest_float("C", 1e-3, 100, log=True)}
    if kind == "tree":
        return {
            "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
            "max_depth": trial.suggest_int("max_depth", 3, 24),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 16),
            "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-6, 1e-2, log=True),
        }
    if kind == "spam":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 60, 160, step=20),
            "num_leaves": trial.suggest_int("num_leaves", 7, 31),
            "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.15, log=True),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 40),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
        }
    raise ValueError(f"Unknown model: {kind}")

In [ ]:
def tune_model(kind: str, X_train, y_train, config: ExperimentConfig):
    cv = StratifiedKFold(n_splits=config.folds, shuffle=True, random_state=config.seed)

    def objective(trial):
        model = make_model(kind, suggest_params(trial, kind), config.seed)
        with threadpool_limits(limits=2):
            scores = cross_val_score(
                model,
                X_train,
                y_train,
                cv=cv,
                scoring="f1",
                n_jobs=1,
                error_score="raise",
            )
        return float(scores.mean())

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=config.seed),
    )
    study.optimize(objective, n_trials=config.trials, n_jobs=1)
    model = make_model(kind, study.best_params, config.seed)
    with threadpool_limits(limits=2):
        model.fit(X_train, y_train)
    return model, study

In [ ]:
config = ExperimentConfig(seed=101, trials=8, folds=3, max_false_positive_rate=0.01)
optuna.logging.set_verbosity(optuna.logging.WARNING)
phishing_split = split_indices(phishing.label, config.seed)
email_split = split_indices(emails.label, config.seed)
X_phishing = phishing.loc[:, list(FEATURE_NAMES)]
y_phishing = phishing.label
X_email = emails.text
y_email = emails.label
display(
    pd.DataFrame(
        {
            "phishing": {k: len(v) for k, v in vars(phishing_split).items()},
            "enron1": {k: len(v) for k, v in vars(email_split).items()},
        }
    ).T
)

## 3. ロジスティック回帰・決定木の探索と学習

In [ ]:
models, studies = {}, {}
for kind in ("logistic", "tree"):
    models[kind], studies[kind] = tune_model(
        kind,
        X_phishing.iloc[phishing_split.train],
        y_phishing.iloc[phishing_split.train],
        config,
    )
    print(f"{kind}: CV F1 = {studies[kind].best_value:.4f}, params = {studies[kind].best_params}")
selected_phishing = max(("logistic", "tree"), key=lambda name: studies[name].best_value)
print(f"CV-selected phishing model: {selected_phishing}")

## 4. TF-IDF + LightGBMの探索と学習

In [ ]:
models["spam"], studies["spam"] = tune_model(
    "spam",
    X_email.iloc[email_split.train],
    y_email.iloc[email_split.train],
    config,
)
print(f"spam: CV F1 = {studies['spam'].best_value:.4f}, params = {studies['spam'].best_params}")

## 5. 検証データで閾値を決める

検証データ上の誤検知率が1%以下になる点のうち、再現率が最大の閾値を採用する。
同点なら高い閾値を使う。陽性判定は `score >= threshold`。
これは検証標本上の制約であり、テストや実運用の誤検知率を保証しない。
スコアは較正した確率ではない。閾値を決めた後はモデルを再学習しない。

In [ ]:
import json
from dataclasses import asdict

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

In [ ]:
def select_threshold(y, scores, max_fpr: float) -> float:
    y, scores = np.asarray(y), np.asarray(scores)
    if not 0 <= max_fpr <= 1:
        raise ValueError("max_fpr must lie in [0, 1]")
    if set(np.unique(y)) != {0, 1} or not np.isfinite(scores).all():
        raise ValueError("Threshold selection requires both classes and finite scores")
    fpr, tpr, thresholds = roc_curve(y, scores, drop_intermediate=False)
    eligible = np.flatnonzero(fpr <= max_fpr)
    # ROC thresholds descend; ties keep the more conservative operating point.
    best = eligible[np.argmax(tpr[eligible])]
    threshold = thresholds[best]
    return float(threshold if np.isfinite(threshold) else np.nextafter(scores.max(), np.inf))

In [ ]:
def binary_metrics(y, scores, threshold: float = 0.5) -> dict:
    prediction = np.asarray(scores) >= threshold
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y, prediction)),
        "precision": float(precision_score(y, prediction, zero_division=0)),
        "recall": float(recall_score(y, prediction, zero_division=0)),
        "f1": float(f1_score(y, prediction, zero_division=0)),
        "average_precision": float(average_precision_score(y, scores)),
        "roc_auc": float(roc_auc_score(y, scores)),
        "false_positive_rate": float(fp / (tn + fp)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

In [ ]:
def feature_importance(model, feature_names=None) -> pd.Series:
    classifier = model.named_steps["classifier"]
    if "tfidf" in model.named_steps:
        feature_names = model.named_steps["tfidf"].get_feature_names_out()
    if hasattr(classifier, "coef_"):
        values = np.abs(classifier.coef_[0])
    else:
        values = classifier.feature_importances_
    return pd.Series(values, index=feature_names, name="importance").sort_values(ascending=False)

In [ ]:
def save_report(output_dir, config, audits, studies, splits, metrics, hashes, revision):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    report = {
        "config": asdict(config),
        "positive_class": {"phishing": "UCI Result = -1", "spam": "enron1/spam"},
        "upstream_revision": revision,
        "dataset_sha256": hashes,
        "data_audit": {name: asdict(audit) for name, audit in audits.items()},
        "split_indices": {
            name: {k: v.tolist() for k, v in vars(s).items()} for name, s in splits.items()
        },
        "search": {
            name: {"best_cv_f1": study.best_value, "params": study.best_params}
            for name, study in studies.items()
        },
        "test_metrics": metrics,
        "versions": {
            name: version(name)
            for name in ("numpy", "pandas", "scipy", "scikit-learn", "lightgbm", "optuna")
        },
    }
    (output_dir / "report.json").write_text(
        json.dumps(report, indent=2, ensure_ascii=False, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    for name, study in studies.items():
        study.trials_dataframe().to_csv(output_dir / f"{name}_trials.csv", index=False)
    return report

In [ ]:
tasks = {
    "logistic": (X_phishing, y_phishing, phishing_split),
    "tree": (X_phishing, y_phishing, phishing_split),
    "spam": (X_email, y_email, email_split),
}
thresholds, validation_metrics = {}, {}
for name, (X, y, split) in tasks.items():
    scores = models[name].predict_proba(X.iloc[split.validation])[:, 1]
    thresholds[name] = select_threshold(
        y.iloc[split.validation],
        scores,
        config.max_false_positive_rate,
    )
    validation_metrics[name] = binary_metrics(y.iloc[split.validation], scores, thresholds[name])
columns = ["threshold", "precision", "recall", "false_positive_rate"]
display(pd.DataFrame(validation_metrics).T[columns])

## 6. 最終テスト

通常の閾値0.5と検証で決めた閾値を併記する。平均適合率（AP）はスコアの順位全体を評価する。
比較用の多数派クラス分類器も訓練データだけで学習する。
テスト結果を使ってモデル・探索範囲・閾値を選び直さない。

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay

test_metrics, test_scores = {}, {}
for name, (X, y, split) in tasks.items():
    scores = models[name].predict_proba(X.iloc[split.test])[:, 1]
    test_scores[name] = scores
    test_metrics[f"{name}/default"] = binary_metrics(y.iloc[split.test], scores)
    test_metrics[f"{name}/validation_threshold"] = binary_metrics(
        y.iloc[split.test],
        scores,
        thresholds[name],
    )
    baseline = DummyClassifier(strategy="most_frequent").fit(
        np.zeros((len(split.train), 1)),
        y.iloc[split.train],
    )
    baseline_scores = baseline.predict_proba(np.zeros((len(split.test), 1)))[:, 1]
    test_metrics[f"{name}/majority"] = binary_metrics(y.iloc[split.test], baseline_scores)
summary = pd.DataFrame(test_metrics).T
display(
    summary[
        [
            "accuracy",
            "precision",
            "recall",
            "f1",
            "average_precision",
            "false_positive_rate",
            "fp",
            "fn",
        ]
    ].round(4)
)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for axis, (name, (_, y, split)) in zip(axes, tasks.items(), strict=True):
    ConfusionMatrixDisplay.from_predictions(
        y.iloc[split.test],
        test_scores[name] >= thresholds[name],
        labels=[0, 1],
        display_labels=["benign", "malicious"],
        colorbar=False,
        ax=axis,
    )
    axis.set_title(name)
fig.tight_layout()
output_dir = Path("artifacts") / "results"
output_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(output_dir / "confusion_matrices.png", bbox_inches="tight")
plt.show()

fig, axis = plt.subplots(figsize=(6, 4))
for name, (_, y, split) in tasks.items():
    PrecisionRecallDisplay.from_predictions(
        y.iloc[split.test],
        test_scores[name],
        name=name,
        ax=axis,
    )
fig.tight_layout()
fig.savefig(output_dir / "precision_recall.png", bbox_inches="tight")
plt.show()

## 7. 特徴量とデータ由来の偏り

ロジスティック回帰は標準化後の係数の絶対値、決定木は不純度の減少、LightGBMはgainを表示する。
重要度は因果関係を示さず、モデルをまたいで値の大きさを比較できない。
書籍が注目する `subject` の出現数も訓練データだけで確認する。
この単語の偏りだけで検知回避が可能とは結論できない。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, (name, model) in zip(axes, models.items(), strict=True):
    importance = feature_importance(model, FEATURE_NAMES if name != "spam" else None)
    importance.to_csv(output_dir / f"{name}_importance.csv", header=True)
    importance.head(10).sort_values().plot.barh(ax=axis, title=name)
fig.tight_layout()
fig.savefig(output_dir / "feature_importance.png", bbox_inches="tight")
plt.show()

email_train = emails.iloc[email_split.train]
subject_audit = (
    email_train.assign(
        subject_count=email_train.text.str.lower().str.count(r"\bsubject\b"),
    )
    .groupby("label")
    .agg(
        messages=("text", "size"),
        subject_tokens=("subject_count", "sum"),
        messages_with_subject=("subject_count", lambda values: int(values.gt(0).sum())),
    )
)
display(subject_audit.rename(index={0: "ham", 1: "spam"}))

## 8. 推論と結果の保存

以下は手入力した英語テキストの推論例。判定の正しさを保証するものではない。
学習データは古く、送信者・時系列・キャンペーンの独立性も確認できない。
重複排除後も近似重複やデータセット特有の偏りは残る。現代の実メールや日本語メールへの性能は未検証。
フィッシングの30特徴量には現在取得しにくいものも含まれるため、実運用には別途設計が必要。

In [ ]:
import shutil

messages = pd.Series(
    [
        "Please review the attached project agenda before tomorrow's meeting.",
        "Congratulations! Claim your free prize now. Limited offer, click to collect.",
    ]
)
scores = models["spam"].predict_proba(messages)[:, 1]
display(
    pd.DataFrame(
        {
            "text": messages,
            "spam_score": scores,
            "predicted_spam": scores >= thresholds["spam"],
        }
    )
)

report = save_report(
    output_dir,
    config,
    {"phishing": phishing_audit, "enron1": email_audit},
    studies,
    {"phishing": phishing_split, "enron1": email_split},
    test_metrics,
    DATA_HASHES,
    UPSTREAM_REVISION,
)
summary.to_csv(output_dir / "test_metrics.csv")
subject_audit.to_csv(output_dir / "subject_audit.csv")
(output_dir / "selected_models.json").write_text(
    json.dumps({"phishing": selected_phishing, "spam": "spam", "thresholds": thresholds}, indent=2),
    encoding="utf-8",
)
archive_path = shutil.make_archive("security-ml-results", "zip", output_dir)
print(f"Results: {archive_path}")

必要なら次のセルでZIPを手元に保存する。学習済みpickleやメール本文は含めず、
指標、分割インデックス、探索履歴、重要度、図、実行バージョンを記録する。

In [ ]:
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    from google.colab import files

    files.download(archive_path)

## 出典

- [書籍サンプル・正誤情報](https://github.com/oreilly-japan/ml-security-jp)
- [UCI Phishing Websites](https://doi.org/10.24432/C51W2X) — R. Mohammad, F. Thabtah, L. McCluskey
- [Enron-Spam](https://www2.aueb.gr/users/ion/data/enron-spam/)
  — V. Metsis, I. Androutsopoulos, G. Paliouras (2006)

§2.5のLing-Spam演習は本ノートブックの対象外。